# Site-Specific MDX Training Tutorial

This notebook provides a guide to the pretraining and site-specific finetuning workflow in the `mdx` repository, as described in [[1]](#ref-1) and [[2]](#ref-2). It explains how to pretrain an MDX on synthetic channels, prepare a measured-data TFRecord for finetuning, finetune the pretrained model with real-world data, and evaluate it on a deterministic set of real-world measurements.

The command cells below describe operations that can be computationally expensive. They are intentionally not executed automatically, and only the commands themselves are printed. Run these commands in a shell rather than from the notebook. Avoid other GPU-intensive workloads while the scripts are running, as it was observed that GPU load can affect the results.

## Contents

- [1. Prerequisites and repository paths](#prerequisites)
- [2. The configuration parameters](#configuration)
- [3. Pretrain the MDX on synthetic channels](#pretraining)
- [4. Prepare the finetuning TFRecord](#tfrecord-preparation)
- [5. Finetune with the Data Lake Channel](#finetuning)
- [6. Evaluate on real-world data](#evaluation)
- [7. Examine the results](#results)
- [References](#references)

<a id="prerequisites"></a>

## 1. Prerequisites and Repository Paths

The required packages are listed in [`requirements.txt`](../requirements.txt). Check [`README.md`](../README.md) for the complete information.

Real-data steps require a running ClickHouse-backed Aerial Data Lake, databases containing the required `fapi` and `fh` tables, and NVIDIA pyAerial. The databases are not included in the repository; detailed information and download instructions are provided in [`datasets.md`](../../datasets.md).

In [ ]:
from pathlib import Path
import os

candidates = (Path.cwd(), *Path.cwd().parents)
WORKSPACE_ROOT = next((path.resolve() for path in candidates if path.name == 'rx_training' and (path / 'mdx' / 'config').is_dir() and (path / 'mdx' / 'scripts').is_dir()), None)
if WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate the rx_training workspace root.')

os.chdir(WORKSPACE_ROOT)
REPO = Path('mdx')
DISPLAY_ROOT = Path('/rx_training')

print(f'Workspace root: {DISPLAY_ROOT}')
print(f'Repository: {DISPLAY_ROOT / REPO}')

Workspace root: /rx_training
Repository: /rx_training/mdx


<a id="configuration"></a>

## 2. The Configuration Parameters

Every experiment is driven by a file in `config/`. Its `global.label` becomes the prefix for weight checkpoints stored as `weights/<label>_weights.h5`. The pretraining configuration uses randomized 3GPP UMi channels implemented in Sionna and matches the 13-symbol resource grid used for the measured PUSCH data. Real-data finetuning uses `channel_type = 'Datalake'` and sets `datalake_tf_fn` to the appropriate TFRecord.

The configuration must match the measured resource-grid dimensions, MCS, DMRS settings, number of receive antennas, and number of active uplink layers. An uplink layer (ULL) denotes one active spatial transmission layer; the supported measured-data workflow uses single-layer transmission (1 ULL).

The tables in [`datasets.md`](../../datasets.md) and the [MDX README](../README.md) list the configurations used in the experiments. All finetuning configurations select `channel_type = 'Datalake'` and point `datalake_tf_fn` to the corresponding TFRecord under `/rx_training/finetuning_datasets/`.

The following code snippet selects the MDX pretraining configuration with the 3GPP UMi channel model and the finetuning configuration for the Jun. 2026 Small Laboratory single-layer dataset.

In [2]:
from mdx.notebooks.notebook_helpers import NotebookHelpers

notebook = NotebookHelpers(WORKSPACE_ROOT, DISPLAY_ROOT)

pretraining_config = 'mdx_res_blocks2_var_mcs_it1_ext_aerial_nomcs.cfg'
finetuning_config = 'mdx_datalake_j61_1ULL.cfg'

pretraining_config_path = REPO / 'config' / pretraining_config
finetuning_config_path = REPO / 'config' / finetuning_config

pretraining_label = notebook.config_value(pretraining_config_path, 'label')
finetuning_label = notebook.config_value(finetuning_config_path, 'label')
finetuning_dataset = notebook.config_value(finetuning_config_path, 'datalake_tf_fn')
transfer_weights_path = notebook.config_value(finetuning_config_path, 'transfer_weights_path')

notebook.show_config(pretraining_config_path)
notebook.show_config(finetuning_config_path)

--- /rx_training/mdx/config/mdx_res_blocks2_var_mcs_it1_ext_aerial_nomcs.cfg

label = 'mdx_res_blocks2_var_mcs_it1_ext_aerial_nrxPe_nomcs' # all relevant files such as weights will use this label
mcs_index = [14]
symbol_allocation = [0, 13]
dmrs_mapping_type = "B"
dmrs_additional_position = 2
max_num_tx = 2 # max number of active DMRS ports
transfer_weights_path = '../weights/mdx_res_blocks2_var_mcs_it1_ext_aerial_nrxPe_weights_nomcs.h5'
channel_type = 'UMi'

--- /rx_training/mdx/config/mdx_datalake_j61_1ULL.cfg

label = 'mdx_datalake_j61_1ULL' # all relevant files such as weights will use this label
mcs_index = [14]
symbol_allocation = [0, 13]
dmrs_mapping_type = "B"
dmrs_additional_position = 2
max_num_tx = 1 # max number of active DMRS ports
transfer_weights_path = '../weights/mdx_res_blocks2_var_mcs_it1_ext_aerial_nrxPe_nomcs_weights.h5'
channel_type = 'Datalake'
datalake_tf_fn = '../../finetuning_datasets/sgs23_1ULLs_5dB_j61_2026_06_04_cell_id_51_no.tfrecord' # tfrecord created fr

<a id="pretraining"></a>

## 3. Pretrain the MDX on Synthetic Channels

Pretraining initializes the model-driven receiver before site-specific adaptation. Start with a pretraining configuration such as `mdx_res_blocks2_var_mcs_it1_ext_aerial_nomcs.cfg`, inspect its training schedule and system parameters, and run [`train_neural_rx.py`](../scripts/train_neural_rx.py) with `-system mdx`.

Initialization follows three steps: load `transfer_weights_path` when that file exists; otherwise resume the checkpoint for `global.label` when present; otherwise start from random weights. Comment out `transfer_weights_path` and ensure no label checkpoint exists for a true from-scratch run.

Training saves the weights to [`/rx_training/mdx/weights/`](../weights/) as `<label>_weights.h5` and the logs to [`/rx_training/mdx/logs/`](../logs/). Depending on the hardware configuration, pretraining might take a long time. The weights used for this experiment are provided, so pretraining can be skipped when reproducing the workflow.

After the pretraining is done, all the finetuning experiments can be done using the same weights, with no need for any additional pretraining.

In [3]:
pretrained_weights = REPO / 'weights' / f'{pretraining_label}_weights.h5'
pretraining_command = f'''
cd {notebook.display_path(REPO / 'scripts')}
python train_neural_rx.py -system mdx -config_name {pretraining_config}
'''.strip()

notebook.show_terminal(
    pretraining_command,
    [('Weights to be saved at', notebook.display_path(pretrained_weights))],
)
# os.system(pretraining_command)

```bash
$ cd /rx_training/mdx/scripts
$ python train_neural_rx.py -system mdx -config_name mdx_res_blocks2_var_mcs_it1_ext_aerial_nomcs.cfg
```

```text
Weights to be saved at: /rx_training/mdx/weights/mdx_res_blocks2_var_mcs_it1_ext_aerial_nrxPe_nomcs_weights.h5
```

<a id="tfrecord-preparation"></a>

## 4. Prepare the Finetuning TFRecord

MDX finetuning uses an existing Data Lake-derived TFRecord, which can be created using [`/neural_rx/scripts/create_tfrecord_from_data_multicell.py`](../../neural_rx/scripts/create_tfrecord_from_data_multicell.py), pr the provided files can be directly used. Place the matching pre-extracted TFRecord to [`/finetuning_datasets`](../../finetuning_datasets/), and specify the path through the `datalake_tf_fn` parameter of the configuration. The TFRecords contain the received IQ samples, coded-bit labels, LS channel estimates, and noise-variance estimates required by the MDX Data Lake channel.

In [5]:
dataset_path = notebook.resolve(REPO / 'scripts' / finetuning_dataset)

print(f'Configured TFRecord: {notebook.display_path(dataset_path)}')
print(f'Exists: {dataset_path.is_file()}')

Configured TFRecord: /rx_training/finetuning_datasets/sgs23_1ULLs_5dB_j61_2026_06_04_cell_id_51_no.tfrecord
Exists: False


<a id="finetuning"></a>

## 5. Finetune with the Data Lake Channel

Finetuning reuses [`train_neural_rx.py`](../scripts/train_neural_rx.py) with `-system mdx`. The difference between the pretraining and the finetuning is the configuration: for finetuning it must select `channel_type = 'Datalake'`, and `datalake_tf_fn` must point to the TFRecord from the previous step. The configuration's `transfer_weights_path` selects the pretrained `.h5` checkpoint, while `global.label` determines the finetuned output checkpoint name.

Before running, verify the following fields in the configuration:

```ini
[global]
label = 'mdx_datalake_j61_1ULL'

[transfer_learning]
transfer_weights_path = '../weights/mdx_res_blocks2_var_mcs_it1_ext_aerial_nrxPe_nomcs_weights.h5'

[training]
channel_type = 'Datalake'
datalake_tf_fn = '../../finetuning_datasets/sgs23_1ULLs_5dB_j61_2026_06_04_cell_id_51_no.tfrecord'
```

The total finetuning duration is set by `training_schedule['num_iter']`, while `num_iter_train_save` controls the regular save interval. For the example configuration, training runs for 100,000 iterations.

Each run writes training logs under [`/rx_training/mdx/logs/`](../logs/). These logs can be inspected with TensorBoard to follow training and compare runs. The finetuned weights are stored as `/rx_training/mdx/weights/<label>_weights.h5`.

In [7]:
finetuned_weights = REPO / 'weights' / f'{finetuning_label}_weights.h5'

finetuning_command = f'''
cd {notebook.display_path(REPO / 'scripts')}
python train_neural_rx.py -system mdx -config_name {finetuning_config}
'''.strip()

notebook.show_terminal(
    finetuning_command,
    [
        ('Transfer weights', notebook.display_path(REPO / 'scripts' / transfer_weights_path)),
        ('Finetuning dataset', notebook.display_path(dataset_path)),
        ('Weights to be saved at', notebook.display_path(finetuned_weights)),
    ],
)
# os.system(finetuning_command)

```bash
$ cd /rx_training/mdx/scripts
$ python train_neural_rx.py -system mdx -config_name mdx_datalake_j61_1ULL.cfg
```

```text
Transfer weights: /rx_training/mdx/weights/mdx_res_blocks2_var_mcs_it1_ext_aerial_nrxPe_nomcs_weights.h5
Finetuning dataset: /rx_training/finetuning_datasets/sgs23_1ULLs_5dB_j61_2026_06_04_cell_id_51_no.tfrecord
Weights to be saved at: /rx_training/mdx/weights/mdx_datalake_j61_1ULL_weights.h5
```

<a id="evaluation"></a>

## 6. Evaluate the Model on Real-World Data

The [`/rx_training/mdx/scripts/eval_mdx_from_datalake_TF.py`](../scripts/eval_mdx_from_datalake_TF.py) script uses ClickHouse to retrieve the selected slot samples and pyAerial for reference-receiver processing. It loads the weights named by the selected configuration label and reports the dataset BLER of the MMSE Reference Rx and the MDX. The MMSE Reference Rx corresponds to the pyAerial PUSCH Rx presented in [[3]](#ref-3).

For repeatable experiments, pass the filename of a timestamp pickle stored in [`/rx_training/mdx/eval_timestamps/`](../eval_timestamps/). The evaluator then replays the same slot samples instead of selecting a random sample set. Omit `--timestamps` when a random selection is intended, or provide a new non-existant filename to save the new timestamps. Choose the database and timestamp file from the tables in the [MDX README](../README.md) and [`datasets.md`](../../datasets.md). For the Jun. 2026 Small Laboratory single-layer example, `--db 9` selects the 1ULL Samsung Galaxy S23 measurement for the Indoor Lab environment. Finetuning and evaluation use temporally disjoint slots from the same UE.

Dataset BLER is the fraction of transport blocks that fail after offline processing and LDPC decoding on a fixed test dataset. It is not system-level BLER under link adaptation.

For each evaluation iteration the script queries slot samples from the `fh` and `fapi` tables, retrieves their transmission parameters, and builds a receiver that matches those parameters. The same samples are also provided to the pyAerial MMSE Reference Rx for a direct comparison. Output is printed to the console, and the aggregate experiment log is stored in `/rx_training/mdx/results/results.txt`.

In [8]:
database = 9
timestamps = 'j61_1ULL.pkl'
evaluation_results = REPO / 'results' / 'results.txt'

evaluation_command = f'''
cd {notebook.display_path(REPO / 'scripts')}
python eval_mdx_from_datalake_TF.py --config-name {finetuning_config} --db {database} --ue 0 --timestamps {timestamps}
'''.strip()

notebook.show_terminal(
    evaluation_command,
    [('Expected results log', notebook.display_path(evaluation_results))],
)
# os.system(evaluation_command)

```bash
$ cd /rx_training/mdx/scripts
$ python eval_mdx_from_datalake_TF.py --config-name mdx_datalake_j61_1ULL.cfg --db 9 --ue 0 --timestamps j61_1ULL.pkl
```

```text
Expected results log: /rx_training/mdx/results/results.txt
```

<a id="results"></a>

## 7. Examine the Results

Site-specific finetuning improves the MDX without increasing inference complexity. On the Jun. 2026 single-layer measurements, finetuning reduces the dataset BLER by about one third in the Small Laboratory scenario and by nearly one half on the Large Office Floor [[2]](#ref-2). The MMSE Reference Rx still achieves a lower dataset BLER than the finetuned MDX in both environments, while the MDX retains a much smaller tunable-parameter footprint than a fully trainable neural receiver.

These gains demonstrate that adaptation to measured propagation conditions and hardware effects benefits the model-driven architecture as well. Repeat the finetuning and evaluation steps with `mdx_datalake_jfloor_1ULL.cfg` and `jfloor_1ULL.pkl` to reproduce the Large Office Floor comparison documented in the [MDX README](../README.md).

<a id="references"></a>

## References

<a id="ref-1"></a>[1] M. Abdollahpour, M. Bertuletti, Y. Zhang, Y. Li, L. Benini, and A. Vanelli-Coralli, “A Compute&Memory Efficient Model-Driven Neural 5G Receiver for Edge AI-assisted RAN,” in *Proc. IEEE Global Communications Conference (GLOBECOM)*, 2025, pp. 5248–5253. Available: https://arxiv.org/abs/2508.12892

<a id="ref-2"></a>[2] R. Wiesmayr, N. B. Baytekin, C. Dick, and C. Studer, “On the Impact of Site-Specific Training for a Real-World 5G NR System,” in *Proc. Asilomar Conference on Signals, Systems, and Computers*, 2026.

<a id="ref-3"></a>[3] NVIDIA Corporation, “Aerial CUDA-Accelerated RAN,” release 25-2. Available: https://docs.nvidia.com/aerial/cuda-accelerated-ran/25-2/index.html